# Implementación de GRPO y KTO. 

1. Primero creamos los conjuntos de datos de alineación. Tienen que seguir el formato correspondiente para cada algoritmo.
2. Posteriormente implementamos los algoritmos de alineación usando el mítico TRL.

Vamos a trabajar solamente con el mejor checkpoint de cada familia de modelos. Esto nos deja con 4 modelos en total:
1. Gemma-3-12B
2. DeepSeek-R1-0528-Qwen3-8B
3. Qwen3-14B-FP8
4. gpt-oss-20B

In [2]:
import pandas as pd
#from huggingface_hub import login
from datasets import load_dataset, Dataset, concatenate_datasets

fol_train = pd.read_json('/home/flopezp/Kurosagol/FOLIO/FOLIO/folio_train.jsonl', lines = True)
#kto_ds = load_dataset("trl-lib/kto-mix-14k", split="train")
grpo_ds = load_dataset('trl-lib/DeepMath-103K', split = 'train')

### KTO Dataset

In [3]:
#Prompts

translation_p = """
    Given a set of premises, the task is to parse the problem and the question into first-order logic formulars. Answer only with the translated premises.
    The grammar of the first-order logic formular is defined as follows:
    1) logical conjunction of expr1 and expr2: expr1 ∧ expr2
    2) logical disjunction of expr1 and expr2: expr1 ∨ expr2
    3) logical exclusive disjunction of expr1 and expr2: expr1 ⊕ expr2
    4) logical negation of expr1: ¬expr1
    5) expr1 implies expr2: expr1 → expr2
    6) expr1 if and only if expr2: expr1 ↔ expr2
    7) logical universal quantification: ∀x
    8) logical existential quantification: ∃x
    --------------
    Natural Language Premises:
    All people who regularly drink coffee are dependent on caffeine. People either regularly drink coffee or joke about being addicted to caffeine. No one who jokes about being addicted to caffeine is unaware that caffeine is a drug. Rina is either a student and unaware that caffeine is a drug, or neither a student nor unaware that caffeine is a drug. If Rina is not a person dependent on caffeine and a student, then Rina is either a person dependent on caffeine and a student, or neither a person dependent on caffeine nor a student.
    Predicates:
    Dependent(x) ::: x is a person dependent on caffeine.
    Drinks(x) ::: x regularly drinks coffee.
    Jokes(x) ::: x jokes about being addicted to caffeine.
    Unaware(x) ::: x is unaware that caffeine is a drug.
    Student(x) ::: x is a student.
    FOL Premises:
    ∀x (Drinks(x) → Dependent(x)) ::: All people who regularly drink coffee are dependent on caffeine.
    ∀x (Drinks(x) ⊕ Jokes(x)) ::: People either regularly drink coffee or joke about being addicted to caffeine.
    ∀x (Jokes(x) → ¬Unaware(x)) ::: No one who jokes about being addicted to caffeine is unaware that caffeine is a drug. 
    (Student(rina) ∧ Unaware(rina)) ⊕ ¬(Student(rina) ∨ Unaware(rina)) ::: Rina is either a student and unaware that caffeine is a drug, or neither a student nor unaware that caffeine is a drug. 
    ¬(Dependent(rina) ∧ Student(rina)) → (Dependent(rina) ∧ Student(rina)) ⊕ ¬(Dependent(rina) ∨ Student(rina)) ::: If Rina is not a person dependent on caffeine and a student, then Rina is either a person dependent on caffeine and a student, or neither a person dependent on caffeine nor a student.
    --------------
    
    Natural Language Premises:
    {}
    FOL Premises:
"""


inference_p = """
    Given a set of premises and conclusion in first order logic, your task is to determine the logical validity of the conclusion: True, False, or Uncertain. Answer only with the logical value.
    A True conclusion is one that can be obtained via a valid inference procedure from the given premises.
    A False conclusion is one that contradicts one or more premises during the inference procedure. 
    An Uncertain conclusion is neither True nor False. Meaning that there is insufficient information in the premises to infer it, but the conclusion it self doesn't contradict any premise.
    --------------
    The following example shows a set of premises and conclusions where each conclusion represents a different logical validity. You should answer similarly.
    FOL-PREMISES:
    ∀x (WorkAt(x, meta) → HighIncome(x))
    ∀x (HighIncome(x) → ¬MeansToDestination(x, bus))
    ∀x (MeansToDestination(x, bus) ⊕ MeansToDestination(x, drive))
    ∀x (HaveCar(x) → MeansToDestination(x, drive))
    ∀x (Student(x) → ¬ MeansToDestination(x, drive))
    HaveCar(james) ∨ WorkAt(james, meta)
    --------------
    FOL-CONCLUSION:
    MeansToDestination(x, drive) ∨ Student(james)
    Student(james)
    ¬HighIncome(james)

    Analysis:
    The first conclusion is True. Premise 6 states that either James has a car (in which case premise 4 gives us the conclusion) or James works at Meta (in which case premise 4 implies premise 2, which combined with premise 3 gives us the conclusion)
    The second conclusion is False. Premise 5 states that students can't have a Car as a MeansToDestination, however the first condition tells us James has such means.
    The third contition is Uncertain. Premise 1 is the only guarantee to have a High Income, however we can't determine that James works at Meta (Premise 6).
    ----------------------------
    FOL-PREMISES:
    {}
    --------------
    FOL-CONCLUSION:
    {}
    --------------
    ANSWER:
"""

retranslation_p = """
    Given a single premise in first order logic, your task is to translate the premise into natural language. Answer only with the translated premise. It should be a single sentence.
    The grammar of the first-order logic formular is defined as follows:
    1) logical conjunction of expr1 and expr2: expr1 ∧ expr2
    2) logical disjunction of expr1 and expr2: expr1 ∨ expr2
    3) logical exclusive disjunction of expr1 and expr2: expr1 ⊕ expr2
    4) logical negation of expr1: ¬expr1
    5) expr1 implies expr2: expr1 → expr2
    6) expr1 if and only if expr2: expr1 ↔ expr2
    7) logical universal quantification: ∀x
    8) logical existential quantification: ∃x
    --------------
    Below are examples of the translation:
    PREMISES:
    ¬(PartTime(jackie) ⊕ ForbesList(jackie)) → ∃y (LessThan(y, num2) ∧ TakesCourses(x,y)) ∧ ForbesList(jackie)
    ¬In(borjMasouda, tunisia)

    NATURAL LANGUAGE:
    If Jackie either enrolls as part-time in the current semester and is listed in the Forbes 30 Under 30, or neither enrolls as part-time in the current semester nor is listed in the Forbes 30 Under 30, then Jackie takes less than two courses in the current semester and listed in the Forbes 30 Under 30.
    Borj Masouda is not in Tunisia.
    --------------    
    PREMISE:
    {}

    NATURAL LANGUAGE:
"""

**KTO Datasets**

In [4]:
# Nmms está todo culero el código cabrón, me emperra que se vea tan feo.
def create_good_dataset(path):
    length = len(fol_train['premises-FOL'])

    prompts, completions, labels = [], [], []
    for i in range(length):    
        translation = fol_train['premises'][i]
        neuro_symb = fol_train['premises-FOL'][i]
        neuro_label = fol_train['label'][i]
        neuro_conc = fol_train['conclusion-FOL'][i]
        retrans = fol_train['conclusion'][i]

        prompts.append([{'role': 'user', 'content': translation_p.format(translation)}]) 
        prompts.append([{'role': 'user', 'content': inference_p.format(neuro_symb, neuro_conc)}])
        prompts.append([{'role': 'user', 'content': retranslation_p.format(neuro_conc)}])

        completions.append([{'role': 'assistant', 'content': neuro_symb}])
        completions.append([{'role': 'assistant', 'content': str(neuro_label)}])
        completions.append([{'role': 'assistant', 'content': retrans}])

        labels.append(True)
        labels.append(True)
        labels.append(True)
    
    pos_dataset = Dataset.from_dict({'prompt': prompts, 'completion': completions, 'label': labels})

    kto_ds = pd.read_csv(path)
    kto_ds = kto_ds.drop(columns = ['Unnamed: 0'])

    ds_name = path.split('/')[-1][:-4]

    trans_p = [[{'role':'user', 'content':translation_p.format(fol_train['premises'][i])}] for i in range(0, len(fol_train['premises']))]
    infer_p = [[{'role':'user', 'content': inference_p.format(fol_train['premises-FOL'][i], fol_train['conclusion-FOL'][i])}] for i in range(0, len(fol_train['premises']))]
    retr_p = [[{'role':'user', 'content': retranslation_p.format(fol_train['conclusion-FOL'][i])}] for i in range(0, len(fol_train['premises']))]
    comps = [[{'role': 'assistant', 'content': str(kto_ds['Full'][i])}] for i in range(len(kto_ds['Full']))]

    prompts = trans_p + infer_p + retr_p
    labels = [False for i in range(len(comps))]

    neg_dataset = Dataset.from_dict({'prompt': prompts, 'completion': comps, 'label': labels})

    concat_ds = concatenate_datasets([pos_dataset, neg_dataset])
    print('KTO Dataset: {}'.format(ds_name))

    concat_ds.push_to_hub('Kurosawama/{}'.format(ds_name), private = True)
    return concat_ds

In [11]:
paths = [
    '/home/flopezp/Kurosagol/Ongoing/alignment_datasets/KTO_DeepSeek-R1-0528-Qwen3-8B.csv',
    '/home/flopezp/Kurosagol/Ongoing/alignment_datasets/KTO_gemma-3-12b-it.csv',
    '/home/flopezp/Kurosagol/Ongoing/alignment_datasets/KTO_gpt-oss-20b.csv',
    '/home/flopezp/Kurosagol/Ongoing/alignment_datasets/KTO_Qwen3-14B-FP8.csv'
]

# KTO Datasets chidos.
for path in paths:
    aux = create_good_dataset(path)

KTO Dataset: KTO_DeepSeek-R1-0528-Qwen3-8B


Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

...

KTO Dataset: KTO_Qwen3-14B-FP8


Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

In [ ]:
#Obtaining Negative Answers (KTO)

from vllm import LLM, SamplingParams

checkpoint_list = [
    ['Qwen/Qwen3-14B-FP8', 'fp8'], 
    ['google/gemma-3-12b-it','fp8'], 
    ['openai/gpt-oss-20b', 'mxfp4'], 
    ['deepseek-ai/DeepSeek-R1-0528-Qwen3-8B', None]
]

def vllm_generation(model_id, quant):
    """
        Generación iterativa sobre distintos modelos usando vLLM.
    """
    print(f"Iniciando con: {model_id}...")
    
    trans_prompts = [translation_p.format(fol_train['premises'][i]) for i in range(len(fol_train['premises']))]
    infer_prompts = [inference_p.format(fol_train['premises-FOL'][i], fol_train['conclusion-FOL'][i]) for i in range(len(fol_train['premises-FOL']))]
    retrans_prompts = [retranslation_p.format(fol_train['conclusion-FOL'][i]) for i in range(len(fol_train['conclusion-FOL']))]

    prompts = trans_prompts + infer_prompts + retrans_prompts
    sampling_params = SamplingParams(temperature = 0.2, top_p = 0.9, max_tokens = 4000, seed = 67, logprobs = 1)
    llm = LLM(
        model = model_id, 
        max_model_len = 6000, 
        quantization = quant, 
        enforce_eager = True, 
        gpu_memory_utilization = 0.9, 
        limit_mm_per_prompt={"image": 0, "video": 0}
    )
    outputs = llm.generate(prompts, sampling_params)

    step_list = []
    #perp_list = []
    i = 0
    for output in outputs:
        gen_text = output.outputs[0].text.strip(r'\n')
        step_list.append(gen_text)
        #loglist = output.outputs[0].logprobs
        #logit_values = [loglist[i].get(list(loglist[i].keys())[0]).logprob for i in range(len(loglist))]
        #perp_list.append(get_perplexity(logit_values))
        if i % 20 == 0:
            print('Iteración: {}'.format(i))
        i += 1

        
    # To do: Pensar en otras cosas que medirle a las generaciones de los modelos.
    df_name = model_id.split('/')[1] # AHHHHHHHHHHHHHHHHHHHHHHH
    full_dataframe = pd.DataFrame({f"Full": step_list})
    full_dataframe.to_csv(f'/home/flopezp/Kurosagol/Ongoing/alignment_datasets/KTO_{df_name}.csv')
    print("=" * 60)
    print("=" * 15 +  f" Fin de: {model_id} " + "=" * 15)
    print("=" * 60)

#for _ in checkpoint_list:
#    vllm_generation(_[0], _[1])

Iniciando con: Qwen/Qwen3-14B-FP8...
INFO 05-11 11:55:30 [utils.py:233] non-default args: {'max_model_len': 6000, 'disable_log_stats': True, 'quantization': 'fp8', 'enforce_eager': True, 'limit_mm_per_prompt': {'image': 0, 'video': 0}, 'model': 'Qwen/Qwen3-14B-FP8'}
INFO 05-11 11:56:51 [model.py:533] Resolved architecture: Qwen3ForCausalLM
INFO 05-11 11:56:51 [model.py:1582] Using max model len 6000
INFO 05-11 11:56:52 [scheduler.py:231] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 05-11 11:56:52 [vllm.py:754] Asynchronous scheduling is enabled.
WARNING 05-11 11:56:52 [vllm.py:788] Enforce eager set, disabling torch.compile and CUDAGraphs. This is equivalent to setting -cc.mode=none -cc.cudagraph_mode=none
WARNING 05-11 11:56:52 [vllm.py:799] Inductor compilation was disabled by user settings, optimizations settings that are only active during inductor compilation will be ignored.
INFO 05-11 11:56:52 [vllm.py:964] Cudagraph is disabled under eager mode
INFO 05-11 1

Loading safetensors checkpoint shards:   0% Completed | 0/4 [00:00<?, ?it/s]


(EngineCore pid=966256) INFO 05-11 11:58:30 [default_loader.py:384] Loading weights took 13.55 seconds
(EngineCore pid=966256) INFO 05-11 11:58:31 [gpu_model_runner.py:4566] Model loading took 15.33 GiB memory and 95.236088 seconds
(EngineCore pid=966256) WARNING 05-11 11:58:31 [fp8_utils.py:1174] Using default W8A8 Block FP8 kernel config. Performance might be sub-optimal! Config file not found at /home/flopezp/.venvs/foo/lib/python3.13/site-packages/vllm/model_executor/layers/quantization/utils/configs/N=7168,K=5120,device_name=NVIDIA_RTX_5000_Ada_Generation,dtype=fp8_w8a8,block_shape=[128,128].json
(EngineCore pid=966256) WARNING 05-11 11:58:31 [fp8_utils.py:1174] Using default W8A8 Block FP8 kernel config. Performance might be sub-optimal! Config file not found at /home/flopezp/.venvs/foo/lib/python3.13/site-packages/vllm/model_executor/layers/quantization/utils/configs/N=5120,K=5120,device_name=NVIDIA_RTX_5000_Ada_Generation,dtype=fp8_w8a8,block_shape=[128,128].json
(EngineCore pi

Rendering prompts:   0%|          | 0/3003 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/3003 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s…

...

(EngineCore pid=1151796) Unrecognized keys in `rope_parameters` for 'rope_type'='yarn': {'attn_factor'}


(EngineCore pid=1151796) WARNING 05-11 17:03:57 [vllm.py:788] Enforce eager set, disabling torch.compile and CUDAGraphs. This is equivalent to setting -cc.mode=none -cc.cudagraph_mode=none
(EngineCore pid=1151796) WARNING 05-11 17:03:57 [vllm.py:799] Inductor compilation was disabled by user settings, optimizations settings that are only active during inductor compilation will be ignored.
(EngineCore pid=1151796) INFO 05-11 17:03:57 [vllm.py:964] Cudagraph is disabled under eager mode
INFO 05-11 17:03:57 [llm.py:391] Supported tasks: ['generate']


Rendering prompts:   0%|          | 0/3003 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/3003 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s…

Iteración: 0
Iteración: 20
Iteración: 40
Iteración: 60
Iteración: 80
Iteración: 100
Iteración: 120
Iteración: 140
Iteración: 160
Iteración: 180
Iteración: 200
Iteración: 220
Iteración: 240
Iteración: 260
Iteración: 280
Iteración: 300
Iteración: 320
Iteración: 340
Iteración: 360
Iteración: 380
Iteración: 400
Iteración: 420
Iteración: 440
Iteración: 460
Iteración: 480
Iteración: 500
Iteración: 520
Iteración: 540
Iteración: 560
Iteración: 580
Iteración: 600
Iteración: 620
Iteración: 640
Iteración: 660
Iteración: 680
Iteración: 700
Iteración: 720
Iteración: 740
Iteración: 760
Iteración: 780
Iteración: 800
Iteración: 820
Iteración: 840
Iteración: 860
Iteración: 880
Iteración: 900
Iteración: 920
Iteración: 940
Iteración: 960
Iteración: 980
Iteración: 1000
Iteración: 1020
Iteración: 1040
Iteración: 1060
Iteración: 1080
Iteración: 1100
Iteración: 1120
Iteración: 1140
Iteración: 1160
Iteración: 1180
Iteración: 1200
Iteración: 1220
Iteración: 1240
Iteración: 1260
Iteración: 1280
Iteración: 1300


**GRPO Datasets**

In [12]:
# Bendito seas grpo que tienes el mismo formato jodido caw. Gracias.
trans_p = [[{'role':'user', 'content':translation_p.format(fol_train['premises'][i])}] for i in range(0, len(fol_train['premises']))]
trans_comps = [[{'role': 'assistant', 'content': fol_train['conclusion-FOL'][i]}] for i in range(0, len(fol_train['premises']))]

infer_p = [[{'role':'user', 'content': inference_p.format(fol_train['premises-FOL'][i], fol_train['conclusion-FOL'][i])}] for i in range(0, len(fol_train['premises']))]
infer_comps = [[{'role': 'assistant', 'content': str(fol_train['label'][i])}] for i in range(0, len(fol_train['premises']))]

retr_p = [[{'role':'user', 'content': retranslation_p.format(fol_train['conclusion-FOL'][i])}] for i in range(0, len(fol_train['premises']))]
retr_comps = [[{'role': 'assistant', 'content': str(fol_train['conclusion'][i])}] for i in range(0, len(fol_train['premises']))]

prompts = trans_p + infer_p + retr_p
comps = trans_comps + infer_comps + retr_comps

grpo_ds2 = Dataset.from_dict({'prompt': prompts, 'solution': comps})
grpo_ds.push_to_hub('Kurosawama/GRPO_FOL', private = True)

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

CommitInfo(commit_url='https://huggingface.co/datasets/Kurosawama/GRPO_FOL/commit/8b91c110d872dfa738c998f58cf765ecfafaa421', commit_message='Upload dataset', commit_description='', oid='8b91c110d872dfa738c998f58cf765ecfafaa421', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/Kurosawama/GRPO_FOL', endpoint='https://huggingface.co', repo_type='dataset', repo_id='Kurosawama/GRPO_FOL'), pr_revision=None, pr_num=None)

### Alignment

In [ ]:
# train_grpo.py
from datasets import load_dataset
from trl import GRPOTrainer, GRPOConfig
from trl.rewards import accuracy_reward

dataset = load_dataset("Kurosawama/GRPO_FOL", split = 'train')

config = GRPOConfig(
    loss_type ='dr_grpo',
    use_vllm = True,
    epsilon = 0.15,
)

trainer = GRPOTrainer(
    model="deepseek-ai/DeepSeek-R1-0528-Qwen3-8B",
    reward_funcs=accuracy_reward,
    train_dataset=dataset,
)
trainer.train()

Unrecognized keys in `rope_parameters` for 'rope_type'='yarn': {'attn_factor'}
Unrecognized keys in `rope_parameters` for 'rope_type'='yarn': {'attn_factor'}


Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

Some parameters are on the meta device because they were offloaded to the cpu.
Unrecognized keys in `rope_parameters` for 'rope_type'='yarn': {'attn_factor'}
Unrecognized keys in `rope_parameters` for 'rope_type'='yarn': {'attn_factor'}
The model is already on multiple devices. Skipping the move to device specified in `args`.
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 151645}.
Passing `generation_config` together with generation-related arguments=({'disable_compile'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


OutOfMemoryError: CUDA out of memory. Tried to allocate 192.00 MiB. GPU 0 has a total capacity of 31.60 GiB of which 185.06 MiB is free. Including non-PyTorch memory, this process has 31.40 GiB memory in use. Of the allocated memory 30.80 GiB is allocated by PyTorch, and 241.33 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)